In [1]:
#Importing package
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
#reading files adding id to every file in shape of session 
base_path = Path.cwd().parent
file1 = pd.read_parquet(f"{base_path}/data/raw/data_telemetry_1793987272.parquet")
file2 = pd.read_parquet(f"{base_path}/data/raw/data_telemetry_1794055365.parquet")
file3 = pd.read_parquet(f"{base_path}/data/raw/data_telemetry_1794086898.parquet")
file4 = pd.read_parquet(f"{base_path}/data/raw/data_telemetry_1794138426.parquet")
file1["session"] = 1
file2["session"] = 2
file3["session"] = 3
file4["session"] = 4



In [3]:
#Deleting starting laps and ending ones and also concating 4 files into one dataframe
file1 = file1[(file1["completedLaps"] > 0) & (file1["completedLaps"] < file1["completedLaps"].max())]
file2 = file2[(file2["completedLaps"] > 0) & (file2["completedLaps"] < file2["completedLaps"].max())]
file3 = file3[(file3["completedLaps"] > 0) & (file3["completedLaps"] < file3["completedLaps"].max())]
file4 = file4[(file4["completedLaps"] > 0) & (file4["completedLaps"] < file4["completedLaps"].max())]
df = pd.concat([file1 , file2 , file3 , file4] , ignore_index=True)

In [4]:
#Adding collums for max_tyres min_speed and max_time per lap
df["max_tyres"] = df.groupby(["session" , "completedLaps"])["numberOfTyresOut"].transform("max")
df["min_speed"] = df.groupby(["session" , "completedLaps"])["speedKmh"].transform("min")
df["max_time"] = df.groupby(["session","completedLaps" ])["iCurrentTime"].transform("max")
print(df.head(2))

   packetId  gas  brake  steerAngle    speedKmh  gear  rpms    accG_x  \
0     53372  1.0    0.0    0.024803  229.204117     6  6404 -0.351136   
1     53378  1.0    0.0    0.024806  229.314590     6  6409 -0.362893   

     accG_y    accG_z  ...  iBestTime  status  isInPit  carCoordinates_x  \
0  0.150957  0.175958  ...     161006       2        0       -204.670593   
1  0.007600  0.175316  ...     161006       2        0       -204.642227   

   carCoordinates_y  carCoordinates_z  session  max_tyres  min_speed  max_time  
0         -8.262426        411.833130        1          4  47.058716    116181  
1         -8.258071        410.878326        1          4  47.058716    116181  

[2 rows x 25 columns]


In [5]:
#Calculating time of each clean lap in seconds and deleting the ones that are too long too slow or the ones that car landed on outside of track
df = df[(df["max_tyres"] <= 2) & (df["min_speed"] > 20) & (df["max_time"] < 130000)]
df["cleanLaps"] = df.groupby(["session" , "completedLaps"])["iCurrentTime"].transform("max") / 1000
print(df.shape)


(216567, 26)


In [6]:
#dividing a track into 1000 elements where one element is approximately 5.8 meters long 
df["track_segment"] = pd.cut(df["normalizedCarPosition"] , bins=1000 , labels=False)
#having gears staring from reverse so R=0 , N = 1 ,1st gear =2 ...
df["gear"] = df["gear"] - 1
#making average of each segment of the lap and wrting that to clean_telemetry parquet file
df = df[~((df["normalizedCarPosition"]< 0.01 ) & (df["iCurrentTime"] > 1000))] 
df_segments = df.groupby(["session", "completedLaps" , "track_segment" ])[["speedKmh", "gas", "brake", "steerAngle", "gear", "rpms", "cleanLaps" , "iCurrentTime" , "carCoordinates_x" , "carCoordinates_y" , "carCoordinates_z"]].mean().reset_index()

print(df_segments.head(20))
df_segments.to_parquet(f"{base_path}/data/processed/clean_telemetry.parquet")

    session  completedLaps  track_segment    speedKmh  gas  brake  \
0         1              5              0  228.949947  1.0    0.0   
1         1              5              1  228.823316  1.0    0.0   
2         1              5              2  229.408194  1.0    0.0   
3         1              5              3  229.947507  1.0    0.0   
4         1              5              4  230.514862  1.0    0.0   
5         1              5              5  231.090134  1.0    0.0   
6         1              5              6  231.556999  1.0    0.0   
7         1              5              7  232.059626  1.0    0.0   
8         1              5              8  232.614975  1.0    0.0   
9         1              5              9  233.167676  1.0    0.0   
10        1              5             10  233.709500  1.0    0.0   
11        1              5             11  234.200006  1.0    0.0   
12        1              5             12  234.675998  1.0    0.0   
13        1              5        